# parameter recovery

colors trial type:


yellow = 1

orange = 2

red = 3

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('svg')
matplotlib.rcParams['svg.fonttype'] = 'none'
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns
from scipy.io import loadmat
import ast
from scipy.stats import pearsonr
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FixedLocator
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")

In [2]:
outputFolderName = r"param_recovery_3_simulated_fields_sanity_check"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\context_modeling\param_recovery_3_simulated_fields"


if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)


matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)


In [3]:
for pt in range(nPatients):
# for pt in range(1):

    fileName = matFiles[pt]

    if not isinstance(fileName, str):
        fileName = str(fileName)

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    result = np.asarray(TDdataParamRecovery.result).astype(str).squeeze()
    reward = np.asarray(TDdataParamRecovery.Reward, dtype=float).squeeze()
    isControl = np.asarray(TDdataParamRecovery.is_control).astype(int).squeeze()
    resultSimulated = np.asarray(TDdataParamRecovery.resultSimulated).astype(str).squeeze()
    rewardSimulated = np.asarray(TDdataParamRecovery.rewardSimulated, dtype=float).squeeze()

    # make sure all arrays have same length first
    min_len = min(len(result), len(reward), len(isControl), len(resultSimulated), len(rewardSimulated))

    result = result[:min_len]
    reward = reward[:min_len]
    isControl = isControl[:min_len]
    resultSimulated = resultSimulated[:min_len]
    rewardSimulated = rewardSimulated[:min_len]

    # KEEP ONLY non-control trials
    noncontrol_idx = np.where(isControl == 0)[0]

    if len(noncontrol_idx) == 0:
        print(f"Skipping {ptID}: no isControl == 0 trials")
        continue

    result = result[noncontrol_idx]
    reward = reward[noncontrol_idx]
    resultSimulated = resultSimulated[noncontrol_idx]
    rewardSimulated = rewardSimulated[noncontrol_idx]

    # convert result labels to numeric for plotting
    all_result_labels = np.unique(np.concatenate([result, resultSimulated]))
    label_to_num = {lab: i for i, lab in enumerate(all_result_labels)}

    result_num = np.array([label_to_num[x] for x in result])
    resultSim_num = np.array([label_to_num[x] for x in resultSimulated])

    # figure
    fig, axes = plt.subplots(2, 1, figsize=(5, 5))

    # reward subplot
    line1, = axes[0].plot(np.arange(len(reward)), reward, label='real', linewidth=1)
    line2, = axes[0].plot(np.arange(len(rewardSimulated)), rewardSimulated, label='simulated', linewidth=1, alpha=0.5)
    axes[0].set_title(f"{ptID} - reward")
    axes[0].set_xlabel("trial")
    axes[0].set_ylabel("reward")

    leg0 = axes[0].legend(
        frameon=False,
        handlelength=0,
        handletextpad=0.3,
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0
    )
    for text, line in zip(leg0.get_texts(), [line1, line2]):
        text.set_color(line.get_color())

    # result subplot
    line3, = axes[1].plot(np.arange(len(result_num)), result_num, label='real', linewidth=1)
    line4, = axes[1].plot(np.arange(len(resultSim_num)), resultSim_num, label='simulated', linewidth=1, alpha=0.5)
    axes[1].set_title(f"{ptID} - result")
    axes[1].set_xlabel("trial")
    axes[1].set_ylabel("result")
    axes[1].set_yticks(range(len(all_result_labels)))
    axes[1].set_yticklabels(all_result_labels)

    leg1 = axes[1].legend(
        frameon=False,
        handlelength=0,
        handletextpad=0.3,
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0
    )
    for text, line in zip(leg1.get_texts(), [line3, line4]):
        text.set_color(line.get_color())

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.subplots_adjust(right=0.78)

    save_path_pdf = os.path.join(outputFolderName, f"{ptID}_reward_result_comparison.pdf")
    plt.savefig(save_path_pdf, format='pdf', bbox_inches='tight')

    save_path_svg = os.path.join(outputFolderName, f"{ptID}_reward_result_comparison.svg")
    plt.savefig(save_path_svg, format='svg', bbox_inches='tight')

    plt.close(fig)

processing pt 1/71: 201810
processing pt 2/71: 201811
processing pt 3/71: 201901
processing pt 4/71: 201902r
processing pt 5/71: 201902
processing pt 6/71: 201903
processing pt 7/71: 201905
processing pt 8/71: 201909
processing pt 9/71: 201910
processing pt 10/71: 201911
processing pt 11/71: 201913
processing pt 12/71: 201914
processing pt 13/71: 201915
processing pt 14/71: 202001
processing pt 15/71: 202002
processing pt 16/71: 202003
processing pt 17/71: 202004
processing pt 18/71: 202005
processing pt 19/71: 202006u
processing pt 20/71: 202006
processing pt 21/71: 202007
processing pt 22/71: 202008
processing pt 23/71: 202009
processing pt 24/71: 202011
processing pt 25/71: 202014
processing pt 26/71: 202015
processing pt 27/71: 202016
processing pt 28/71: 202105
processing pt 29/71: 202107
processing pt 30/71: 202110
processing pt 31/71: 202114
processing pt 32/71: 202117
processing pt 33/71: 202118
processing pt 34/71: 202201
processing pt 35/71: 202202
processing pt 36/71: 202205

# debug

In [4]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

['RPE', 'Reward', 'a', 'bestAlphaNeg', 'bestAlphaPos', 'bestAnIdx', 'bestApIdx', 'bestFitScore', 'bestInverseTemperature', 'bestLogLik', 'expectedReward', 'fitScoreRSTD', 'inflate_time', 'inverseTemperatureRSTD', 'is_control', 'logLikRSTD', 'logPriorAlphaNeg', 'logPriorAlphaPos', 'nTrials', 'points', 'pointsMinusReward', 'result', 'resultSimulated', 'rewardSimulated', 'scoreVec', 'trialTypes', 'trial_type']
